# Day 3 — Feature Scaling & Normalisation
**Week 3 · Feature Engineering & Data Prep · AI Engineer Lab**

Scale matters for gradient descent, distance metrics, and PCA. This notebook covers:
- StandardScaler (Z-score)
- MinMaxScaler (bounded [0,1])
- RobustScaler (outlier-resistant)
- Normalizer (row-wise unit norm)
- PowerTransformer / QuantileTransformer (fixing skewness)
- Which algorithms need scaling vs which don't
- The golden rule: fit on train, transform test

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    Normalizer, PowerTransformer, QuantileTransformer
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer

np.random.seed(42)
print('Libraries loaded.')

## 1. Create a Dataset with Realistic Scale Differences

In [ ]:
n = 1000
np.random.seed(42)

df = pd.DataFrame({
    'age':         np.random.randint(20, 70, n).astype(float),          # [20, 70]
    'income':      np.random.exponential(55000, n),                      # [0, 500k], skewed
    'credit_score':np.clip(np.random.normal(680, 80, n), 300, 850),     # [300, 850]
    'num_loans':   np.random.poisson(2.5, n).astype(float),             # [0, ~10], count
    'savings':     np.random.exponential(20000, n),                      # [0, 200k], skewed
})
# Inject outliers into income
df.loc[np.random.choice(n, 20, replace=False), 'income'] = np.random.uniform(400000, 1000000, 20)

# Target: loan default
score = (-0.3 * (df['income'] / 50000) + 0.2 * (df['num_loans'] / 5)
         - 0.4 * (df['credit_score'] / 700) + np.random.randn(n) * 0.5)
df['default'] = (score > 0.1).astype(int)

print('Feature stats:')
print(df.drop(columns='default').describe().round(2))

## 2. Visualise Raw Feature Distributions

In [ ]:
features = ['age', 'income', 'credit_score', 'num_loans', 'savings']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, feat in zip(axes, features):
    ax.hist(df[feat], bins=40, color='#3b82f6', alpha=0.7, edgecolor='none')
    ax.set_title(feat, fontsize=10)
    skew_val = df[feat].skew()
    ax.set_xlabel(f'skew={skew_val:.2f}')

plt.suptitle('Raw Feature Distributions', fontsize=12)
plt.tight_layout()
plt.show()
print('Income and savings are highly right-skewed (skew > 2). Age and credit score are near-normal.')

## 3. Apply All Scalers — Side-by-Side Comparison

In [ ]:
X = df[features].values
y = df['default'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scalers = {
    'StandardScaler':   StandardScaler(),
    'MinMaxScaler':     MinMaxScaler(),
    'RobustScaler':     RobustScaler(),
    'PowerTransformer': PowerTransformer(method='yeo-johnson'),
}

# Fit on train, transform train and test
scaled_data = {}
for name, scaler in scalers.items():
    X_tr_sc = scaler.fit_transform(X_train)
    X_te_sc = scaler.transform(X_test)
    scaled_data[name] = (X_tr_sc, X_te_sc)

# Plot 'income' column after each scaling
income_idx = 1  # income is column 1
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

axes[0].hist(X_train[:, income_idx], bins=40, color='#94a3b8', alpha=0.8)
axes[0].set_title('Raw Income', fontsize=10)

for ax, (name, (X_tr, _)) in zip(axes[1:], scaled_data.items()):
    ax.hist(X_tr[:, income_idx], bins=40, color='#3b82f6', alpha=0.8)
    ax.set_title(name, fontsize=9)

plt.suptitle('Income Column After Different Scalers', fontsize=12)
plt.tight_layout()
plt.show()

## 4. RobustScaler vs StandardScaler — Outlier Effect

In [ ]:
# Create toy dataset with one extreme outlier
x_normal = np.random.normal(0, 1, 100)
x_outlier = np.append(x_normal, [50, 60, -40])  # extreme outliers

std_scaled = StandardScaler().fit_transform(x_outlier.reshape(-1, 1)).flatten()
rob_scaled = RobustScaler().fit_transform(x_outlier.reshape(-1, 1)).flatten()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(x_outlier, bins=30, color='#94a3b8', alpha=0.8)
axes[0].set_title(f'Raw (with outliers)\nrange=[{x_outlier.min():.0f}, {x_outlier.max():.0f}]')

axes[1].hist(std_scaled, bins=30, color='#3b82f6', alpha=0.8)
axes[1].set_title(f'StandardScaler\nrange=[{std_scaled.min():.1f}, {std_scaled.max():.1f}]')
axes[1].set_xlabel('Most values compressed to tiny range!')

axes[2].hist(rob_scaled, bins=30, color='#22c55e', alpha=0.8)
axes[2].set_title(f'RobustScaler\nrange=[{rob_scaled.min():.1f}, {rob_scaled.max():.1f}]')
axes[2].set_xlabel('Normal values well spread, outliers isolated')

plt.suptitle('Effect of Outliers on Scaling', fontsize=12)
plt.tight_layout()
plt.show()

## 5. PowerTransformer — Fix Skewed Distributions

In [ ]:
income_raw = df['income'].values.reshape(-1, 1)
pt = PowerTransformer(method='yeo-johnson')
income_pt = pt.fit_transform(income_raw).flatten()

qt = QuantileTransformer(output_distribution='normal', random_state=42)
income_qt = qt.fit_transform(income_raw).flatten()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(income_raw, bins=50, color='#94a3b8', alpha=0.8)
axes[0].set_title(f'Raw Income (skew={df["income"].skew():.2f})')

axes[1].hist(income_pt, bins=50, color='#a78bfa', alpha=0.8)
from scipy import stats
axes[1].set_title(f'Yeo-Johnson Transform (skew={pd.Series(income_pt).skew():.3f})')

axes[2].hist(income_qt, bins=50, color='#22d3ee', alpha=0.8)
axes[2].set_title(f'QuantileTransformer (normal)')

plt.suptitle('Fixing Skewed Distributions', fontsize=12)
plt.tight_layout()
plt.show()
print('PowerTransformer reduces skewness dramatically — makes linear models work better.')

## 6. Impact of Scaling on Model Performance

In [ ]:
models_scaling = {
    'Logistic Regression (no scale)':   LogisticRegression(max_iter=1000),
    'Logistic Regression + Standard':   Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))]),
    'Logistic Regression + Robust':     Pipeline([('s', RobustScaler()), ('m', LogisticRegression(max_iter=1000))]),
    'KNN (no scale)':                   KNeighborsClassifier(n_neighbors=5),
    'KNN + Standard':                   Pipeline([('s', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=5))]),
    'Random Forest (no scale)':         RandomForestClassifier(n_estimators=50, random_state=42),
    'Random Forest + Standard':         Pipeline([('s', StandardScaler()), ('m', RandomForestClassifier(n_estimators=50, random_state=42))]),
}

results = {}
for name, model in models_scaling.items():
    cv = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
    results[name] = cv.mean()

print('=== Scaling Impact on Model Performance (5-Fold AUC) ===')
for name, auc in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {name:<45} AUC = {auc:.4f}')

print('\nKey finding: Logistic Regression and KNN benefit massively from scaling.')
print('Random Forest is scale-invariant — same AUC with/without scaling.')

## 7. The Golden Rule: Fit on Train, Transform Test

In [ ]:
scaler = StandardScaler()

# CORRECT approach
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test

print('Scaler mean (from training data):', scaler.mean_[:3].round(3))
print('Scaler std (from training data): ', scaler.scale_[:3].round(3))
print()

# WRONG approach — DO NOT DO THIS
# X_test_scaled = scaler.fit_transform(X_test)  # re-fitting on test leaks test stats!
print('WRONG: scaler.fit_transform(X_test) recalculates mean/std from test data.')
print('       This is data leakage — test statistics should never influence training.')
print()
print('CORRECT: fit on X_train only, then .transform() on both X_train and X_test.')

# Verify no leakage: test set transformed using training statistics
print(f'\nTest mean after correct scaling (should be ~0): {X_test_scaled.mean(axis=0)[:3].round(3)}')
print('Not exactly 0 because test statistics ≠ train statistics — that is correct.')

## Exercises

1. **Skewness analysis**: Compute the skewness of all features in the dataset. Apply PowerTransformer (Yeo-Johnson) to features with |skew| > 1. Compare model performance before and after.

2. **Outlier sensitivity**: Manually inject 5 extreme outliers into the `credit_score` feature. Compare how StandardScaler vs RobustScaler handles these on model performance.

3. **Normalizer for text vectors**: Create 1000 random TF-IDF-like vectors (sparse, positive). Apply Normalizer (L2 norm). Verify that each row has unit L2 norm after transformation.

4. **Learning rate experiment**: Train logistic regression with gradient descent (SGD). Measure epochs to convergence with and without StandardScaler. Plot loss curves for both.

In [ ]:
# ── Exercise 3 Solution: Normalizer for TF-IDF vectors ──
from sklearn.preprocessing import Normalizer

# Simulate TF-IDF vectors (sparse, non-negative)
tfidf_vectors = np.abs(np.random.randn(1000, 500)) * (np.random.rand(1000, 500) > 0.85)
print(f'Raw vector norms — mean: {np.linalg.norm(tfidf_vectors, axis=1).mean():.2f}')

normalizer = Normalizer(norm='l2')
tfidf_normalized = normalizer.fit_transform(tfidf_vectors)

norms = np.linalg.norm(tfidf_normalized, axis=1)
print(f'After L2 normalisation — all norms = {norms[:5].round(4)}')
print(f'Min: {norms.min():.4f}, Max: {norms.max():.4f}')
print('All norms = 1.0 → unit vectors → cosine similarity = dot product')